# Análise da Campanha Mínima Viável

Métricas por réplica, critérios de aceitação (Tabela 4.2) e comparações pareadas entre MECADE e baseline.

Para a campanha real, `DATA_ROOT` aponta para `data/runs`. Uso `data/sim/runs` (com `INCLUDE_SIM=True`) só para testar o pipeline, e esses dados simulados não entram em nenhum resultado.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'analysis' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from analysis.metrics import load_runs, metrics_table, paired_comparison
DATA_ROOT = ROOT / 'data' / 'runs'
INCLUDE_SIM = False
FIG = ROOT / 'analysis' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
runs = load_runs(DATA_ROOT, include_sim=INCLUDE_SIM)
m = metrics_table(runs)
print(len(runs), 'réplicas carregadas'); m.head()

## 1. Visão geral por cenário e braço

In [ ]:
cols = ['detected','TTD','TTM','TTR_capped','MTTR','area','TSR_fault','p99_fault','RRIndex_run']
main = m[m['pair'].astype(str) != '']
main.groupby(['scenario','arm'])[cols].median().round(3)

## 2. Classificação tripartite (Tabela 3.5) e detecção

In [ ]:
pd.crosstab([main['scenario'], main['arm']], main['classe'])

In [ ]:
main.groupby(['scenario','arm']).agg(n=('run_id','count'), detectadas=('detected','sum'), ttr_censuradas=('TTR_censored','sum'), rto_ok=('rto_ok','sum'))

## 3. Comparações pareadas (MECADE menos baseline)

Wilcoxon pareado, delta de Cliff e IC de 95% por *bootstrap* da mediana das diferenças. Nas réplicas sem restauração, `TTR_capped` usa o fim da janela, o que é conservador.

In [ ]:
pd.concat([paired_comparison(m, k) for k in ['TTR_capped','area','TSR_fault','p99_fault','RRIndex_run']], ignore_index=True).round(4)

## 4. Taxa de falsos bloqueios (réplicas de controle)

In [ ]:
ctrl = m[m['pair'].astype(str) == '']
print(f"Controles: {len(ctrl)} | BLOCK acionado em {int(ctrl['block'].sum())} "
      f"=> taxa de falsos bloqueios = {ctrl['block'].mean() if len(ctrl) else float('nan'):.2%}")

## 5. Overhead da auditoria (Camada 6, perfil mínimo)

Δ_audit no caminho de decisão é o p95 do custo de enfileirar o evento. Comparo também a CPU do controlador com auditoria ligada e desligada.

In [ ]:
aud = m[m['arm']=='mecade'].groupby('audit')[['audit_p95_ms','controller_cpu_s']].median()
print(aud)
print('Critério: Δ_audit,p95 <= 5 ms ->', bool((m['audit_p95_ms'].dropna() <= 5).all()))

## 6. Figuras

In [ ]:
def fig_series(scenario):
    fig, ax = plt.subplots(figsize=(7, 3.6), dpi=150)
    for meta, df in runs:
        if meta['scenario'] != scenario or not meta.get('pair'):
            continue
        d = df[df['t'] >= 0]
        ax.plot(d['t'], d['D_agr'], lw=0.8, alpha=0.6, color='tab:red' if meta['arm']=='mecade' else 'tab:gray')
    ax.axvline(0, ls=':', c='k'); ax.set_xlabel('t desde a injeção (s)'); ax.set_ylabel('D_agr')
    ax.set_title(f'{scenario}: MECADE (vermelho) × baseline (cinza)')
    fig.tight_layout(); fig.savefig(FIG / f'Dagr_{scenario}.png', dpi=300)
for sc in sorted(main['scenario'].unique()):
    fig_series(sc)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6), dpi=150)
main.boxplot(column='TTR_capped', by=['scenario','arm'], ax=ax, rot=30)
ax.set_ylabel('TTR (s)'); plt.suptitle(''); fig.tight_layout(); fig.savefig(FIG / 'TTR_box.png', dpi=300)

## 7. Exportação para a dissertação

In [ ]:
m.to_csv(ROOT / 'analysis' / 'metricas_por_replica.csv', index=False)
print('analysis/metricas_por_replica.csv gravado')